In [ ]:
from pathlib import Path
from io import BytesIO
import json
 
import numpy as np
import matplotlib.pyplot as plt
 
from matplotlib.colors import BoundaryNorm, ListedColormap
from PIL import Image as PILImage
from IPython.display import display, Image as DisplayImage, Markdown
 
 

In [ ]:
 
# ============================================================
# Configuration
# ============================================================
 
CHIP_DIR = Path(
    "/Users/jacaraba/Desktop/development/satvision-pix4d/abi-cloudsat-crop-test-inner-disk-2500"
)
 
N_CHIPS = 6
ABI_BAND = 13
FPS = 1.0
 
# "cloudiest", "random", or "first"
SELECTION = "cloudiest"
RANDOM_SEED = 42
 


In [ ]:

# ============================================================
# Visualization helpers
# ============================================================
 
cloud_colors = [
    "#777777",  # -1: missing retrieval
    "#ffffff",  #  0: valid clear
    "#66c2ff",  #  1
    "#33a02c",  #  2
    "#e31a1c",  #  3
    "#9467bd",  #  4
    "#8c564b",  #  5
    "#e377c2",  #  6
    "#bcbd22",  #  7
    "#17becf",  #  8
]
 
cloud_cmap = ListedColormap(cloud_colors)
cloud_norm = BoundaryNorm(
    np.arange(-1.5, 9.5, 1),
    cloud_cmap.N,
)
 
 
def load_chip(path):
    with np.load(path) as source:
        sample = {
            name: source[name]
            for name in source.files
        }
 
    sample["metadata"] = json.loads(
        str(sample.pop("metadata_json"))
    )
 
    return sample
 
 
def cloudiness_score(path):
    with np.load(path) as source:
        if "cloudsat_cloud_mask" not in source.files:
            return -1.0
 
        mask = source["cloudsat_cloud_mask"]
 
        valid_profiles = np.any(mask >= 0, axis=1)
        cloudy_profiles = np.any(mask > 0, axis=1)
 
        if not np.any(valid_profiles):
            return -1.0
 
        return float(
            np.mean(cloudy_profiles[valid_profiles])
        )
 
 
def track_distance_km(latitude, longitude):
    latitude = np.deg2rad(latitude)
    longitude = np.deg2rad(longitude)
 
    if len(latitude) < 2:
        return np.zeros(len(latitude))
 
    delta_latitude = np.diff(latitude)
    delta_longitude = np.diff(longitude)
 
    haversine = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(latitude[:-1])
        * np.cos(latitude[1:])
        * np.sin(delta_longitude / 2) ** 2
    )
 
    segments = (
        2
        * 6371.0
        * np.arcsin(np.sqrt(haversine))
    )
 
    return np.concatenate([
        [0.0],
        np.cumsum(segments),
    ])
 
 
def band_limits(chip, band_index):
    values = chip[..., band_index]
    valid = values[np.isfinite(values)]
 
    if valid.size == 0:
        return 0.0, 1.0
 
    lower, upper = np.percentile(valid, [2, 98])
 
    if np.isclose(lower, upper):
        upper = lower + 1.0
 
    return float(lower), float(upper)
 
 
def draw_abi(axis, sample, frame, band_index, limits):
    band = sample["chip"][frame, :, :, band_index]
 
    lower, upper = limits
    normalized = np.clip(
        (band - lower) / (upper - lower),
        0,
        1,
    )
 
    # Invert infrared radiance so cold cloud tops appear bright.
    image = 1.0 - normalized
    image[~np.isfinite(band)] = np.nan
 
    cmap = plt.get_cmap("gray").copy()
    cmap.set_bad("magenta")
 
    axis.imshow(
        image,
        cmap=cmap,
        vmin=0,
        vmax=1,
        origin="upper",
    )
 
    metadata = sample["metadata"]
 
    if (
        "cloudsat_abi_row" in sample
        and "cloudsat_abi_column" in sample
    ):
        height, width = band.shape
 
        track_y = (
            sample["cloudsat_abi_row"]
            - metadata["abi_row"]
            + height / 2
        )
 
        track_x = (
            sample["cloudsat_abi_column"]
            - metadata["abi_column"]
            + width / 2
        )
 
        inside = (
            (track_x >= 0)
& (track_x < width)
& (track_y >= 0)
& (track_y < height)
        )
 
        axis.plot(
            track_x[inside],
            track_y[inside],
            color="red",
            linewidth=2.5,
        )
 
        if np.any(inside):
            x = track_x[inside]
            y = track_y[inside]
 
            axis.scatter(
                x[0],
                y[0],
                color="cyan",
                s=35,
                zorder=3,
            )
 
            axis.scatter(
                x[-1],
                y[-1],
                color="yellow",
                s=35,
                zorder=3,
            )
 
    else:
        axis.text(
            0.5,
            0.03,
            "No saved CloudSat ABI coordinates",
            transform=axis.transAxes,
            ha="center",
            color="yellow",
            bbox={
                "facecolor": "black",
                "alpha": 0.7,
            },
        )
 
    offset = int(
        sample["abi_offsets_minutes"][frame]
    )
    scan_time = str(
        sample["abi_scan_times"][frame]
    )
    valid = int(
        sample["abi_valid_mask"][frame]
    )
 
    axis.set_title(
        f"ABI C{ABI_BAND:02d} | "
        f"{offset:+d} min | valid={valid}\n"
        f"{scan_time}"
    )
 
    axis.set_xlabel("ABI column")
    axis.set_ylabel("ABI row")
 
 
def draw_cloudsat_mask(axis, sample):
    if "cloudsat_cloud_mask" not in sample:
        axis.text(
            0.5,
            0.5,
            "CloudSat mask not saved",
            ha="center",
            va="center",
        )
        axis.axis("off")
        return
 
    mask = sample["cloudsat_cloud_mask"]
 
    distance = track_distance_km(
        sample["cloudsat_latitude"],
        sample["cloudsat_longitude"],
    )
 
    maximum_distance = max(
        float(distance[-1]),
        1e-6,
    )
 
    axis.imshow(
        mask.T,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        extent=[
            0,
            maximum_distance,
            0,
            mask.shape[1] * 0.5,
        ],
        cmap=cloud_cmap,
        norm=cloud_norm,
    )
 
    if "cloudsat_profile_valid" in sample:
        valid_profiles = (
            sample["cloudsat_profile_valid"]
            .astype(bool)
        )
    else:
        # Legacy files do not distinguish missing and clear.
        valid_profiles = np.any(
            mask >= 0,
            axis=1,
        )
 
    cloudy_profiles = np.any(
        mask > 0,
        axis=1,
    )
 
    if np.any(valid_profiles):
        valid_fraction = np.mean(valid_profiles)
        cloudy_fraction = np.mean(
            cloudy_profiles[valid_profiles]
        )
 
        title = (
            f"CloudSat mask\n"
            f"valid={valid_fraction:.1%}, "
            f"cloudy={cloudy_fraction:.1%}"
        )
    else:
        title = "CloudSat mask\nno valid retrievals"
 
    axis.set_title(title)
    axis.set_xlabel("Distance along track (km)")
    axis.set_ylabel("Altitude (km)")
    axis.set_ylim(0, 20)
 
 
def draw_cloudsat_metadata(axis, sample, frame):
    required = {
        "cloudsat_cloud_layer_base",
        "cloudsat_cloud_layer_top",
        "cloudsat_cloud_layer_type",
    }
 
    if not required.issubset(sample):
        axis.text(
            0.5,
            0.5,
            "CloudSat layer metadata not saved",
            ha="center",
            va="center",
        )
        axis.axis("off")
        return
 
    distance = track_distance_km(
        sample["cloudsat_latitude"],
        sample["cloudsat_longitude"],
    )
 
    bases = sample[
        "cloudsat_cloud_layer_base"
    ]
    tops = sample[
        "cloudsat_cloud_layer_top"
    ]
    types = sample[
        "cloudsat_cloud_layer_type"
    ]
 
    for profile in range(len(distance)):
        for layer in range(bases.shape[1]):
            cloud_type = int(
                types[profile, layer]
            )
 
            if (
                bases[profile, layer] < 0
                or tops[profile, layer] < 0
                or cloud_type <= 0
            ):
                continue
 
            color_index = min(
                cloud_type + 1,
                len(cloud_colors) - 1,
            )
 
            axis.vlines(
                distance[profile],
                bases[profile, layer],
                tops[profile, layer],
                color=cloud_colors[color_index],
                linewidth=2,
            )
 
    metadata = sample["metadata"]
 
    if "cloudsat_profile_valid" in sample:
        valid_fraction = np.mean(
            sample["cloudsat_profile_valid"]
            .astype(bool)
        )
        valid_text = f"{valid_fraction:.1%}"
    else:
        valid_text = "legacy/unknown"
 
    summary = (
        f"orbit: {metadata.get('cloudsat_orbit')}\n"
        f"center lat: "
        f"{metadata.get('center_latitude', np.nan):.2f}\n"
        f"center lon: "
        f"{metadata.get('center_longitude', np.nan):.2f}\n"
        f"profiles: {len(distance)}\n"
        f"valid: {valid_text}\n"
        f"frame: {frame + 1}/"
        f"{sample['chip'].shape[0]}"
    )
 
    axis.text(
        0.02,
        0.98,
        summary,
        transform=axis.transAxes,
        va="top",
        bbox={
            "facecolor": "white",
            "alpha": 0.85,
        },
    )
 
    axis.set_title("CloudSat layer metadata")
    axis.set_xlabel("Distance along track (km)")
    axis.set_ylabel("Layer altitude (km)")
    axis.set_ylim(0, 20)
    axis.grid(alpha=0.25)
 
 
def make_gif(path):
    sample = load_chip(path)
 
    band_index = ABI_BAND - 1
    limits = band_limits(
        sample["chip"],
        band_index,
    )
 
    frames = []
 
    for frame in range(
        sample["chip"].shape[0]
    ):
        figure, axes = plt.subplots(
            1,
            3,
            figsize=(18, 5.5),
            dpi=90,
        )
 
        draw_abi(
            axes[0],
            sample,
            frame,
            band_index,
            limits,
        )
 
        draw_cloudsat_mask(
            axes[1],
            sample,
        )
 
        draw_cloudsat_metadata(
            axes[2],
            sample,
            frame,
        )
 
        figure.suptitle(
            path.name,
            fontsize=10,
        )
 
        figure.tight_layout(
            rect=(0, 0, 1, 0.95)
        )
 
        figure.canvas.draw()
 
        image = PILImage.fromarray(
            np.asarray(
                figure.canvas.buffer_rgba()
            )
        ).convert("RGB")
 
        frames.append(image)
        plt.close(figure)
 
    buffer = BytesIO()
 
    frames[0].save(
        buffer,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=int(1000 / FPS),
        loop=0,
        disposal=2,
        optimize=False,
    )
 
    return buffer.getvalue()


In [ ]:
 
 
# ============================================================
# Select N chips
# ============================================================
 
chip_files = sorted(
    CHIP_DIR.glob("*.npz")
)
 
if not chip_files:
    raise FileNotFoundError(
        f"No NPZ files found in {CHIP_DIR}"
    )
 
if SELECTION == "cloudiest":
    chip_files = sorted(
        chip_files,
        key=cloudiness_score,
        reverse=True,
    )
    selected_files = chip_files[:N_CHIPS]
 
elif SELECTION == "random":
    rng = np.random.default_rng(
        RANDOM_SEED
    )
 
    count = min(
        N_CHIPS,
        len(chip_files),
    )
 
    indices = rng.choice(
        len(chip_files),
        size=count,
        replace=False,
    )
 
    selected_files = [
        chip_files[index]
        for index in indices
    ]
 
elif SELECTION == "first":
    selected_files = chip_files[:N_CHIPS]
 
else:
    raise ValueError(
        "SELECTION must be "
        "'cloudiest', 'random', or 'first'"
    )
 

In [ ]:

 
# ============================================================
# Create and display GIFs directly in the notebook
# ============================================================
 
print(
    f"Displaying {len(selected_files)} "
    f"of {len(chip_files)} chips"
)
 
for chip_number, path in enumerate(
    selected_files,
    start=1,
):
    score = cloudiness_score(path)
 
    display(
        Markdown(
            f"### Chip {chip_number}/"
            f"{len(selected_files)}\n"
            f"`{path.name}`\n\n"
            f"Cloudy-profile fraction: "
            f"**{score:.1%}**"
        )
    )
 
    gif_data = make_gif(path)
 
    display(
        DisplayImage(
            data=gif_data,
            format="gif",
        )
    )